# Module 5 — Current-Season Deployment

This notebook refreshes the pipeline against live data and generates a current-form ranking for the in-progress 2026–27 season. The model is re-trained on five full seasons (2021–22 through 2025–26) rather than re-using the Module 3 artifact, because league-wide fantasy averages drift year over year with changes in pace, three-point rates, and rotations. Re-training on the full available history is the correct default when the underlying distribution may have shifted.

Early-season predictions are noisier than in-season predictions because fewer games have been played, rotations are still being set, and rookies have minimal rolling history. The ranker therefore applies a five-game minimum rather than the thirty-game minimum used in Module 4.

**Output:** a top-thirty current-form ranker, produced from aggregated per-game predictions, intended for weekly waiver and lineup decisions in my live fantasy league. MAE on the 2026–27 games played so far is 10.54 FP, approximately four percent worse than the 2024–25 backtest, within the expected variance given the small live sample.

## Step 1 — Pull fresh data from nba_api

`LeagueGameLog` returns one row per (player, game). One API call per season = very fast.

In [6]:
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import leaguegamelog

# Pull the two new seasons we don't have yet
new_seasons = ['2025-26', '2026-27']
new_dfs = []

for season in new_seasons:
    print(f'Fetching {season}...')
    gl = leaguegamelog.LeagueGameLog(
        season=season,
        player_or_team_abbreviation='P',
        season_type_all_star='Regular Season',
    ).get_data_frames()[0]
    gl['SEASON'] = season
    gl['GAME_DATE'] = pd.to_datetime(gl['GAME_DATE'])
    new_dfs.append(gl)
    print(f'  {gl.shape[0]:,} rows, date range {gl["GAME_DATE"].min().date()} to {gl["GAME_DATE"].max().date()}')

new_data = pd.concat(new_dfs, ignore_index=True)
print(f'\nTotal new rows: {new_data.shape[0]:,}')

Fetching 2025-26...
  26,651 rows, date range 2025-10-21 to 2026-04-12
Fetching 2026-27...
  2,981 rows, date range 2026-10-20 to 2026-11-06

Total new rows: 29,632


## Step 2 — Concatenate with existing 2021-2024 data

In [7]:
# Load existing data
old_data = pd.read_parquet('data/raw/player_games_2021_2024.parquet')
print(f'Existing data: {old_data.shape[0]:,} rows (2021-22 through 2024-25)')

# Align column sets (new data may have slightly different cols)
common_cols = sorted(set(old_data.columns) & set(new_data.columns))
print(f'Common columns: {len(common_cols)}')

# Concat
pg = pd.concat([old_data[common_cols], new_data[common_cols]], ignore_index=True)
pg = pg.sort_values(['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)

print(f'\nExpanded dataset: {pg.shape[0]:,} rows, {pg["SEASON"].nunique()} seasons')
print(f'Date range: {pg["GAME_DATE"].min().date()} to {pg["GAME_DATE"].max().date()}')
print(f'\nSeason breakdown:')
print(pg['SEASON'].value_counts().sort_index())

# Save expanded raw data
pg.to_parquet('data/raw/player_games_2021_2026.parquet', index=False)
print(f'\nSaved: data/raw/player_games_2021_2026.parquet')

Existing data: 104,641 rows (2021-22 through 2024-25)
Common columns: 33

Expanded dataset: 134,273 rows, 6 seasons
Date range: 2021-10-19 to 2026-11-06

Season breakdown:
SEASON
2021-22    26039
2022-23    25895
2023-24    26401
2024-25    26306
2025-26    26651
2026-27     2981
Name: count, dtype: int64

Saved: data/raw/player_games_2021_2026.parquet


## Step 3 — Re-run feature engineering on expanded data

Replays all Module 2 logic in one block. Same formulas, more data. Rolling features automatically bridge season boundaries, so a player's rolling_fp_10 in their 2nd game of 2026-27 uses their last 10 games (most from 2025-26).

In [8]:
# ============ FP (user_league scoring) ============
pg['FP'] = (
    pg['PTS']  *  1 + pg['FGM']  *  2 + pg['FGA']  * -1 +
    pg['FTM']  *  1 + pg['FTA']  * -1 + pg['FG3M'] *  1 +
    pg['REB']  *  1 + pg['AST']  *  2 + pg['STL']  *  4 +
    pg['BLK']  *  4 + pg['TOV']  * -2
)

# ============ Sort chronologically per player ============
pg = pg.sort_values(['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)

# ============ Rest days & decomposition ============
pg['rest_days'] = pg.groupby('PLAYER_ID')['GAME_DATE'].diff().dt.days
pg['rest_days_capped'] = pg['rest_days'].clip(upper=10)
pg['returned_from_absence'] = (pg['rest_days'] >= 30).astype(int)

# ============ Rolling features (all leakage-safe via shift(1)) ============
pg['rolling_fp_5']  = pg.groupby('PLAYER_ID')['FP'].transform(lambda s: s.shift(1).rolling(5).mean())
pg['rolling_fp_10'] = pg.groupby('PLAYER_ID')['FP'].transform(lambda s: s.shift(1).rolling(10).mean())
pg['rolling_min_10'] = pg.groupby('PLAYER_ID')['MIN'].transform(lambda s: s.shift(1).rolling(10).mean())
pg['rolling_fp_std_10'] = pg.groupby('PLAYER_ID')['FP'].transform(lambda s: s.shift(1).rolling(10).std())
pg['fp_momentum'] = pg['rolling_fp_5'] - pg['rolling_fp_10']

# ============ Home/away ============
pg['is_home'] = pg['MATCHUP'].str.contains('vs', regex=False).astype(int)

# ============ Opponent defensive strength ============
pg['OPP'] = pg['MATCHUP'].str.split(' ').str[-1]
team_game_fp = (
    pg.groupby(['OPP', 'GAME_DATE'])['FP'].sum()
      .reset_index()
      .rename(columns={'OPP': 'TEAM', 'FP': 'FP_ALLOWED'})
)
team_game_fp = team_game_fp.sort_values(['TEAM', 'GAME_DATE']).reset_index(drop=True)
team_game_fp['FP_ALLOWED_ROLLING_10'] = (
    team_game_fp.groupby('TEAM')['FP_ALLOWED']
                .transform(lambda s: s.shift(1).rolling(10).mean())
)
pg = pg.merge(
    team_game_fp[['TEAM', 'GAME_DATE', 'FP_ALLOWED_ROLLING_10']]
      .rename(columns={'TEAM': 'OPP', 'FP_ALLOWED_ROLLING_10': 'opp_fp_allowed'}),
    on=['OPP', 'GAME_DATE'], how='left',
)

feature_cols = ['rest_days_capped', 'returned_from_absence',
                'rolling_fp_5', 'rolling_fp_10',
                'is_home', 'opp_fp_allowed', 'rolling_min_10',
                'rolling_fp_std_10', 'fp_momentum']

print(f'Expanded feature-engineered data: {pg.shape}')
print(f'Features: {feature_cols}')

Expanded feature-engineered data: (134273, 45)
Features: ['rest_days_capped', 'returned_from_absence', 'rolling_fp_5', 'rolling_fp_10', 'is_home', 'opp_fp_allowed', 'rolling_min_10', 'rolling_fp_std_10', 'fp_momentum']


## Step 4 — Train on 5 full seasons, predict 2026-27 games so far

- **Train:** 2021-22 through 2025-26 (5 complete seasons)
- **Predict:** 2026-27 games (Oct-Nov 2026)

In [10]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

clean_df = pg.dropna(subset=feature_cols).copy()
print(f'After NaN drop: {clean_df.shape[0]:,} rows (was {pg.shape[0]:,})')

# Split by season
train = clean_df[clean_df['SEASON'] != '2026-27'].copy()
test  = clean_df[clean_df['SEASON'] == '2026-27'].copy()

print(f'\nTrain: {train.shape[0]:,} rows  (2021-22 through 2025-26)')
print(f'Test:  {test.shape[0]:,} rows  (2026-27 so far)')

model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    random_state=42, verbose=-1,
)
model.fit(train[feature_cols], train['FP'])

test['predicted_FP'] = model.predict(test[feature_cols])

mae = mean_absolute_error(test['FP'], test['predicted_FP'])
print(f'\n2026-27 MAE (so far): {mae:.2f} FP   (Module 4 MAE was 10.09 on 2024-25)')

After NaN drop: 124,493 rows (was 134,273)

Train: 121,832 rows  (2021-22 through 2025-26)
Test:  2,661 rows  (2026-27 so far)

2026-27 MAE (so far): 10.54 FP   (Module 4 MAE was 10.09 on 2024-25)


## Step 5 — Current-form ranker (YOUR actionable output)

Aggregate per-game predictions into one row per player. Rank by predicted per-game FP = current form.

**Minimum games played = 5** — with only ~9 games played per team so far, being too strict leaves us with nobody. 5 is enough to filter pure noise.

In [11]:
current = test.groupby(['PLAYER_ID', 'PLAYER_NAME']).agg(
    games_played=('GAME_DATE', 'count'),
    actual_avg_fp=('FP', 'mean'),
    predicted_avg_fp=('predicted_FP', 'mean'),
).reset_index()

MIN_GAMES = 5
qualified = current[current['games_played'] >= MIN_GAMES].copy()

ranker = qualified.sort_values('predicted_avg_fp', ascending=False).reset_index(drop=True)
ranker['rank'] = ranker.index + 1

print(f'Qualified players (>= {MIN_GAMES} games in 2026-27): {ranker.shape[0]}')
print(f'\nTop 30 fantasy players RIGHT NOW (as of Nov 6, 2026):\n')
ranker[['rank', 'PLAYER_NAME', 'games_played', 'predicted_avg_fp', 'actual_avg_fp']].head(30)

Qualified players (>= 5 games in 2026-27): 307

Top 30 fantasy players RIGHT NOW (as of Nov 6, 2026):



,rank,PLAYER_NAME,games_played,predicted_avg_fp,actual_avg_fp
0,1,Victor Wembanyama,9,59.895380,64.444444
1,2,Nikola Jokić,7,57.716318,58.285714
2,3,Luka Dončić,9,54.150175,67.000000
3,4,Jayson Tatum,8,45.123596,47.125000
4,5,Cooper Flagg,8,43.192069,48.500000
5,6,Shai Gilgeous-Alexander,9,43.106820,46.777778
6,7,Jamal Murray,7,43.036551,46.428571
7,8,Kevin Durant,9,41.855382,35.555556
8,9,LeBron James,8,41.737906,37.250000
9,10,Devin Booker,7,41.472776,39.428571
